# 06 — Risk Metrics

**Goal:** Quantify the risk-adjusted performance of all six tickers using industry-standard metrics.

DuckDB SQL is the **primary engine** here — every metric, every ranking, every drawdown period is computed with SQL first, with Python only used for the visualisations that SQL cannot produce.

| Metric | What it measures |
|---|---|
| **Sharpe Ratio** | Excess return per unit of total risk (higher = better) |
| **Sortino Ratio** | Excess return per unit of *downside* risk only |
| **Max Drawdown** | Largest peak-to-trough loss (worst case scenario) |
| **Calmar Ratio** | Ann. return / Max Drawdown (higher = better) |
| **VaR 95%** | Worst daily loss in the bottom 5% of days |
| **CVaR 95%** | Average loss *given* you are in that bottom 5% |

Risk-free rate assumed = 4.5% annualised (approx. US T-Bill 2024).

In [1]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

plt.rcParams.update({
    'figure.dpi'       : 150,
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.22,
    'grid.linestyle'   : ':',
})

DATA_DIR  = Path('data')
OUT_DIR   = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

RF_ANNUAL = 0.045          # risk-free rate (annualised)
RF_DAILY  = RF_ANNUAL / 252
COLORS = {
    'AAPL': '#1565C0', 'AMZN': '#E65100', 'GLD' : '#2E7D32',
    'JPM' : '#C62828', 'SPY' : '#6A1B9A', 'XOM' : '#4E342E'
}
print('Imports OK  |  Risk-free rate:', RF_ANNUAL*100, '% p.a.')

Imports OK  |  Risk-free rate: 4.5 % p.a.


In [2]:
# ── 2. Load data ───────────────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'market_data.csv', parse_dates=['Date'])
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
print(f'Loaded: {df.shape}  |  {df.Date.min().date()} → {df.Date.max().date()}')

Loaded: (12426, 10)  |  2018-01-02 → 2026-03-30


## SQL Block 1 — Core Risk Metrics Table

All six metrics computed in a **single SQL query** using window functions and aggregations. This is exactly the kind of query you would write in a real analyst role against a data warehouse.

In [3]:
# ── 3. DuckDB — core risk metrics ─────────────────────────────────────────
RF_D = RF_DAILY

con = duckdb.connect()
con.register("market", df)

risk_table = con.execute("""
    WITH base AS (
        SELECT Ticker, DailyReturn,
               DailyReturn - """ + str(RF_D) + """           AS excess_ret,
               LEAST(DailyReturn - """ + str(RF_D) + """, 0) AS downside_ret
        FROM market
        WHERE DailyReturn IS NOT NULL
    ),
    var_tbl AS (
        SELECT Ticker,
               PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY DailyReturn) AS var_95
        FROM base GROUP BY Ticker
    ),
    stats AS (
        SELECT b.Ticker, COUNT(*) AS n,
               AVG(b.DailyReturn)     AS avg_ret,
               STDDEV(b.DailyReturn)  AS std_ret,
               STDDEV(b.downside_ret) AS downside_std,
               AVG(b.excess_ret)      AS avg_excess,
               v.var_95,
               AVG(CASE WHEN b.DailyReturn <= v.var_95 THEN b.DailyReturn END) AS cvar_95
        FROM base b JOIN var_tbl v ON b.Ticker = v.Ticker
        GROUP BY b.Ticker, v.var_95
    )
    SELECT Ticker,
           ROUND(avg_ret  * 252 * 100,   2)                         AS ann_return_pct,
           ROUND(std_ret  * SQRT(252) * 100, 2)                     AS ann_vol_pct,
           ROUND((avg_excess*252)/(std_ret*SQRT(252)),     3)        AS sharpe_ratio,
           ROUND((avg_excess*252)/(downside_std*SQRT(252)),3)        AS sortino_ratio,
           ROUND(var_95  * 100, 3)                                   AS var_95_pct,
           ROUND(cvar_95 * 100, 3)                                   AS cvar_95_pct
    FROM stats ORDER BY sharpe_ratio DESC
""").df()

print(f"=== Core Risk Metrics (RF = {RF_ANNUAL*100:.1f}% p.a.) ===")
print(risk_table.to_string(index=False))

=== Core Risk Metrics (RF = 4.5% p.a.) ===
Ticker  ann_return_pct  ann_vol_pct  sharpe_ratio  sortino_ratio  var_95_pct  cvar_95_pct
  AAPL           26.74        30.63         0.726          1.206      -2.978       -4.384
   GLD           15.95        16.49         0.694          1.097      -1.572       -2.429
   JPM           18.67        28.94         0.490          0.802      -2.658       -4.143
   SPY           13.83        19.31         0.483          0.746      -1.781       -2.941
  AMZN           20.71        34.32         0.472          0.787      -3.289       -4.906
   XOM           17.61        30.10         0.436          0.722      -2.818       -4.272


## SQL Block 2 — Drawdown Analysis

Max drawdown requires tracking the running **peak** of a cumulative return series, then measuring how far below that peak the price fell. We do this entirely in SQL using window functions.

In [4]:
# ── 4. DuckDB — drawdown per ticker ──────────────────────────────────────
drawdown_sql = con.execute("""
    WITH cum AS (
        SELECT
            Ticker, Date, Close,
            MAX(Close) OVER (
                PARTITION BY Ticker
                ORDER BY Date
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )                                                AS running_peak
        FROM market
    ),
    dd AS (
        SELECT
            Ticker, Date, Close, running_peak,
            (Close - running_peak) / running_peak            AS drawdown
        FROM cum
    )
    SELECT
        Ticker,
        ROUND(MIN(drawdown) * 100, 2)                        AS max_drawdown_pct,
        -- date when max drawdown occurred
        MIN_BY(Date, drawdown)                               AS max_dd_date,
        -- peak date just before the trough
        MIN_BY(running_peak, drawdown)                       AS peak_price
    FROM dd
    GROUP BY Ticker
    ORDER BY max_drawdown_pct ASC
""").df()

print('=== Maximum Drawdown per Ticker ===')
print(drawdown_sql.to_string(index=False))

=== Maximum Drawdown per Ticker ===
Ticker  max_drawdown_pct max_dd_date  peak_price
   XOM            -61.01  2020-03-23   61.511047
  AMZN            -56.15  2022-12-28  186.570496
   JPM            -43.63  2020-03-23  119.036438
  AAPL            -38.52  2019-01-03   54.921650
   SPY            -33.72  2020-03-23  309.197998
   GLD            -22.00  2022-09-26  193.889999


In [5]:
# ── 5. Calmar Ratio = Ann. Return / |Max Drawdown| ────────────────────────
full_metrics = risk_table.merge(
    drawdown_sql[['Ticker','max_drawdown_pct','max_dd_date']], on='Ticker'
)
full_metrics['calmar_ratio'] = (
    full_metrics['ann_return_pct'] / full_metrics['max_drawdown_pct'].abs()
).round(3)

print('=== Full Risk Scorecard ===')
print(full_metrics[['Ticker','ann_return_pct','ann_vol_pct','sharpe_ratio',
                     'sortino_ratio','max_drawdown_pct','calmar_ratio',
                     'var_95_pct','cvar_95_pct']].to_string(index=False))

=== Full Risk Scorecard ===
Ticker  ann_return_pct  ann_vol_pct  sharpe_ratio  sortino_ratio  max_drawdown_pct  calmar_ratio  var_95_pct  cvar_95_pct
  AAPL           26.74        30.63         0.726          1.206            -38.52         0.694      -2.978       -4.384
   GLD           15.95        16.49         0.694          1.097            -22.00         0.725      -1.572       -2.429
   JPM           18.67        28.94         0.490          0.802            -43.63         0.428      -2.658       -4.143
   SPY           13.83        19.31         0.483          0.746            -33.72         0.410      -1.781       -2.941
  AMZN           20.71        34.32         0.472          0.787            -56.15         0.369      -3.289       -4.906
   XOM           17.61        30.10         0.436          0.722            -61.01         0.289      -2.818       -4.272


## SQL Block 3 — Rolling 1-Year Sharpe Ratio

A static Sharpe ratio hides regime changes. We compute a **rolling 252-day Sharpe** to show when each ticker's risk-adjusted performance deteriorated (e.g., during rising-rate periods).

In [6]:
# ── 6. Rolling 1-year Sharpe (pandas, then visualised) ───────────────────
df_sharpe = df.sort_values(['Ticker','Date']).copy()
df_sharpe['excess'] = df_sharpe['DailyReturn'] - RF_DAILY
df_sharpe['RollingSharpe252'] = (
    df_sharpe.groupby('Ticker')['excess']
             .transform(lambda x:
                 (x.rolling(252).mean() * 252) /
                 (x.rolling(252).std()  * np.sqrt(252)))
)

tickers = sorted(df_sharpe['Ticker'].unique())
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=True)
axes = axes.flatten()

for i, ticker in enumerate(tickers):
    ax = axes[i]
    sub = df_sharpe[df_sharpe['Ticker'] == ticker].dropna(subset=['RollingSharpe252'])
    color = COLORS[ticker]

    ax.plot(sub['Date'], sub['RollingSharpe252'], color=color, linewidth=1.4)
    ax.fill_between(sub['Date'], sub['RollingSharpe252'], 0,
                    where=sub['RollingSharpe252'] > 0, alpha=0.18, color=color)
    ax.fill_between(sub['Date'], sub['RollingSharpe252'], 0,
                    where=sub['RollingSharpe252'] < 0, alpha=0.25, color='red')
    ax.axhline(0,   color='black',  linewidth=0.8)
    ax.axhline(1,   color='green',  linewidth=0.8, linestyle='--', alpha=0.6)
    ax.axhline(-0.5, color='red',   linewidth=0.8, linestyle='--', alpha=0.6)
    ax.set_title(ticker, fontweight='bold', fontsize=12, color=color)
    ax.set_ylabel('Rolling Sharpe', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)

fig.suptitle('Rolling 1-Year Sharpe Ratio per Ticker\n'
             '(Green dashed = 1.0 | Red dashed = -0.5 | Shaded red = negative SR)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'rolling_sharpe.png', dpi=150, bbox_inches='tight')
print('Saved → outputs/rolling_sharpe.png')
plt.show()

Saved → outputs/rolling_sharpe.png


## SQL Block 4 — Drawdown Timeline Chart

Visualising the drawdown series (how far below the all-time high each ticker is at any point) makes the COVID crash, 2022 rate-hike cycle, and individual corrections immediately visible.

In [7]:
# ── 7. DuckDB — full drawdown series for plotting ─────────────────────────
dd_series = con.execute("""
    WITH cum AS (
        SELECT
            Ticker, Date, Close,
            MAX(Close) OVER (
                PARTITION BY Ticker
                ORDER BY Date
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS running_peak
        FROM market
    )
    SELECT
        Ticker, Date,
        ROUND((Close - running_peak) / running_peak * 100, 3) AS drawdown_pct
    FROM cum
    ORDER BY Ticker, Date
""").df()
dd_series['Date'] = pd.to_datetime(dd_series['Date'])

fig2, ax2 = plt.subplots(figsize=(14, 6))
for ticker in sorted(dd_series['Ticker'].unique()):
    sub = dd_series[dd_series['Ticker'] == ticker]
    ax2.plot(sub['Date'], sub['drawdown_pct'],
             color=COLORS[ticker], linewidth=1.4, label=ticker, alpha=0.85)

ax2.fill_between(sub['Date'], 0, -100, alpha=0.03, color='red')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axvline(pd.Timestamp('2020-03-20'), color='red', linestyle='--',
            linewidth=1.2, alpha=0.7)
ax2.text(pd.Timestamp('2020-03-20'), -2, ' COVID crash',
         color='red', fontsize=8.5, va='top')

ax2.set_title('Drawdown from All-Time High (%)',
              fontsize=13, fontweight='bold', pad=12)
ax2.set_xlabel('Year', fontsize=11)
ax2.set_ylabel('Drawdown (%)', fontsize=11)
ax2.legend(loc='lower right', fontsize=9, ncol=2)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.xaxis.set_major_locator(mdates.YearLocator())
plt.tight_layout()
plt.savefig(OUT_DIR / 'drawdown_chart.png', dpi=150, bbox_inches='tight')
print('Saved → outputs/drawdown_chart.png')
plt.show()

Saved → outputs/drawdown_chart.png


## SQL Block 5 — Risk Scorecard Heatmap

A seaborn heatmap of normalised risk metrics lets a recruiter (or portfolio manager) instantly see which ticker has the best risk profile on each dimension.

In [8]:
# ── 8. Risk scorecard heatmap ─────────────────────────────────────────────
scorecard = full_metrics.set_index('Ticker')[[
    'ann_return_pct', 'ann_vol_pct', 'sharpe_ratio',
    'sortino_ratio', 'max_drawdown_pct', 'calmar_ratio',
    'var_95_pct', 'cvar_95_pct'
]].copy()
scorecard.columns = [
    'Ann.Return%', 'Ann.Vol%', 'Sharpe', 'Sortino',
    'MaxDD%', 'Calmar', 'VaR95%', 'CVaR95%'
]

# Normalise each column 0-1 for heatmap (higher = better AFTER sign flip for risk cols)
scorecard_norm = scorecard.copy()
for col in ['Ann.Vol%', 'MaxDD%', 'VaR95%', 'CVaR95%']:
    mn, mx = scorecard_norm[col].min(), scorecard_norm[col].max()
    scorecard_norm[col] = 1 - (scorecard_norm[col] - mn) / (mx - mn + 1e-9)
for col in ['Ann.Return%', 'Sharpe', 'Sortino', 'Calmar']:
    mn, mx = scorecard_norm[col].min(), scorecard_norm[col].max()
    scorecard_norm[col] = (scorecard_norm[col] - mn) / (mx - mn + 1e-9)

fig3, axes3 = plt.subplots(1, 2, figsize=(15, 5))

# Left: raw values
sns.heatmap(scorecard, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.4, ax=axes3[0],
            annot_kws={'size': 9})
axes3[0].set_title('Raw Risk Metrics', fontsize=12, fontweight='bold')
axes3[0].set_xlabel('')

# Right: normalised (0=worst, 1=best)
sns.heatmap(scorecard_norm, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0, vmax=1, linewidths=0.4, ax=axes3[1],
            annot_kws={'size': 9})
axes3[1].set_title('Normalised Score (1=best on each metric)', fontsize=12, fontweight='bold')
axes3[1].set_xlabel('')

plt.suptitle('Risk Scorecard — All Tickers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'risk_scorecard.png', dpi=150, bbox_inches='tight')
print('Saved → outputs/risk_scorecard.png')
plt.show()

Saved → outputs/risk_scorecard.png


## SQL Block 6 — Worst Drawdown Periods (Date Ranges)

Identifying *when* the worst drawdown periods started and ended is crucial for correlating with macro events.

In [9]:
# ── 9. DuckDB — worst drawdown periods ───────────────────────────────────
worst_dd_periods = con.execute("""
    WITH cum AS (
        SELECT Ticker, Date, Close,
            MAX(Close) OVER (
                PARTITION BY Ticker ORDER BY Date
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS peak
        FROM market
    ),
    dd AS (
        SELECT Ticker, Date, Close, peak,
            (Close - peak) / peak AS dd_pct
        FROM cum
    ),
    streaks AS (
        SELECT Ticker, Date, dd_pct,
            -- detect when we enter a new drawdown episode
            CASE WHEN dd_pct < 0 AND
                      LAG(dd_pct, 1, 0) OVER (PARTITION BY Ticker ORDER BY Date) = 0
                 THEN 1 ELSE 0 END AS new_episode
        FROM dd
    ),
    episodes AS (
        SELECT Ticker, Date, dd_pct,
            SUM(new_episode) OVER (PARTITION BY Ticker ORDER BY Date) AS ep_id
        FROM streaks
        WHERE dd_pct < 0
    )
    SELECT
        Ticker,
        MIN(Date)                     AS drawdown_start,
        MAX(Date)                     AS drawdown_end,
        ROUND(MIN(dd_pct)*100, 2)     AS max_dd_pct,
        COUNT(*)                      AS trading_days_in_dd
    FROM episodes
    GROUP BY Ticker, ep_id
    HAVING MIN(dd_pct) < -0.10          -- only show >10% drawdowns
    ORDER BY max_dd_pct ASC
    LIMIT 20
""").df()

print('=== 20 Worst Drawdown Periods (>10% peak-to-trough) ===')
print(worst_dd_periods.to_string(index=False))

=== 20 Worst Drawdown Periods (>10% peak-to-trough) ===
Ticker drawdown_start drawdown_end  max_dd_pct  trading_days_in_dd
   XOM     2018-09-25   2022-01-13      -61.01                 833
  AMZN     2021-07-09   2024-04-10      -56.15                 693
   JPM     2020-01-03   2021-01-06      -43.63                 255
   JPM     2021-10-25   2023-12-13      -38.77                 538
  AAPL     2018-10-04   2019-10-09      -38.52                 255
  AMZN     2018-09-05   2020-02-03      -34.10                 355
   SPY     2020-02-20   2020-08-07      -33.72                 119
  AAPL     2024-12-27   2025-10-17      -33.36                 202
  AAPL     2020-02-13   2020-06-04      -31.43                  78
  AAPL     2022-01-04   2023-06-01      -30.91                 354
  AMZN     2025-02-05   2025-10-30      -30.88                 186
   SPY     2022-01-04   2023-12-12      -24.50                 488
   JPM     2025-02-19   2025-06-20      -24.42                  85
  AMZN

In [10]:
print('\n✓ 06_risk_metrics complete')
for f in sorted(OUT_DIR.glob('*.png')):
    print(f'  → {f.name:<42}  {f.stat().st_size/1024:>7.1f} KB')


✓ 06_risk_metrics complete
  → anomaly_AAPL.png                              148.2 KB
  → anomaly_AMZN.png                              165.6 KB
  → anomaly_GLD.png                               137.4 KB
  → anomaly_JPM.png                               142.1 KB
  → anomaly_SPY.png                               146.3 KB
  → anomaly_XOM.png                               144.6 KB
  → correlation_heatmap.png                       104.8 KB
  → drawdown_chart.png                            365.4 KB
  → normalised_price.png                          367.1 KB
  → risk_scorecard.png                            195.5 KB
  → rolling_sharpe.png                            291.9 KB
  → rolling_volatility.png                        352.7 KB
  → spy_forecast.png                              260.7 KB
  → spy_monthly_heatmap.png                       151.9 KB
